<a href="https://colab.research.google.com/github/karan124381/unmasked/blob/main/training/Unmaskd_Colab_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unmaskd — Colab Training Pipeline

This notebook prepares the training environment, downloads the Kaggle video dataset after you authenticate, inspects its actual layout, and prepares the first video training run. It does **not** commit datasets or credentials to GitHub.

Audio training is included as a documented ASVspoof 2021 DF setup because that dataset requires registration/download and should not be silently scraped.


## 0. Runtime

In Colab choose **Runtime → Change runtime type → T4 GPU** if available.


In [1]:
!nvidia-smi || true
!git clone -q https://github.com/karan124381/unmasked.git /content/unmasked
%cd /content/unmasked
!pip install -q -r training/requirements.txt kaggle


Wed Oct  7 23:37:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Kaggle authentication

On your computer, download `kaggle.json` from Kaggle Account → API → Create New Token. Then use the Colab file-upload cell below. Do **not** paste the token into code or commit it.


In [4]:
from google.colab import files
uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'Upload the Kaggle API token named kaggle.json'
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!rm -f kaggle.json
print('Kaggle credentials installed for this Colab runtime.')


Saving kaggle.json to kaggle.json
Kaggle credentials installed for this Colab runtime.


## 2. Download the video dataset

This uses Kaggle's official competition CLI. You must accept the competition rules on Kaggle if prompted. The download can be large, so make sure the Colab runtime has enough storage.


In [6]:
!mkdir -p /content/unmaskd_data/video_raw
!kaggle competitions files deepfake-detection | head -30
!kaggle competitions download deepfake-detection -p /content/unmaskd_data/video_raw --unzip


Next Page Token = CfDJ8LXGgraiXMNClPzcySXY5U7obCYAi4TPCYcWyMpdE-IugLSQwBcg50SfOyg0Gs1kC3VplB6KIkoSxGAX6R2DYz7g-O4XlzRuYxNJocXup03EKIPiluTx4qvca-hys1-pP6GvHMRViPvkRVGIfrBIRigcDESQyUkzleVH73ZYcqMhx9fvNa-j
name                                                 size  creationDate                
---------------------------------------------  ----------  --------------------------  
deep-fake-dataset/Celeb-Youtube-fake/0.mp4        2203086  2021-12-10 13:17:39.099000  
deep-fake-dataset/Celeb-Youtube-fake/1.mp4        1538579  2021-12-10 13:17:39.110000  
deep-fake-dataset/Celeb-Youtube-fake/10.mp4       2139221  2021-12-10 13:17:39.177000  
deep-fake-dataset/Celeb-Youtube-fake/100.mp4      3526883  2021-12-10 13:17:39.210000  
deep-fake-dataset/Celeb-Youtube-fake/1000.mp4     1009221  2021-12-10 13:17:39.202000  
deep-fake-dataset/Celeb-Youtube-fake/1001.mp4     1262289  2021-12-10 13:17:39.191000  
deep-fake-dataset/Celeb-Youtube-fake/1002.mp4      881397  2021-12-10 13:17:39.264000  
deep-

## 3. Inspect the actual Kaggle layout

Do this before assigning labels. Different deepfake datasets use different folder/file conventions; we should never guess that a filename means REAL or FAKE.


In [8]:
!kaggle competitions download deepfake-detection -p /content/unmaskd_data/video_raw

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles


In [9]:
!pip install -U kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.0.2
    Uninstalling kaggle-2.0.2:
      Successfully uninstalled kaggle-2.0.2


In [10]:
!kaggle --version


Kaggle CLI 2.2.4


In [11]:
!kaggle --version

!rm -rf /content/unmaskd_data/video_raw
!mkdir -p /content/unmaskd_data/video_raw

!kaggle competitions download deepfake-detection \
    -p /content/unmaskd_data/video_raw \
    --unzip

Kaggle CLI 2.2.4
usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,forums,f,benchmarks,b,config,auth,quota} ...
kaggle: error: unrecognized arguments: --unzip


In [ ]:
!rm -rf /content/unmaskd_data/video_raw
!mkdir -p /content/unmaskd_data/video_raw

!kaggle competitions download deepfake-detection \
    -p /content/unmaskd_data/video_raw


 77% 7.14G/9.29G [03:38<01:03, 36.5MB/s]

In [ ]:
from pathlib import Path
root = Path('/content/unmaskd_data/video_raw')
files = [p for p in root.rglob('*') if p.is_file()]
print('Files:', len(files))
for p in files[:80]:
    print(p.relative_to(root))


## 4. Prepare a small verified training subset

The existing `prepare_ffpp.py` expects FaceForensics++-style folders. Because the Kaggle competition layout can differ, inspect the output above first. If the dataset contains an explicit metadata/labels file, adapt the preparation step to that metadata. Do not infer labels from arbitrary filenames.


In [ ]:
# Find likely metadata files. Review the printed candidates before using one.
for p in files:
    if p.suffix.lower() in {'.csv','.json','.txt'}:
        print(p.relative_to(root))


### If your downloaded dataset matches the expected REAL/FAKE directory layout

Run the following. Otherwise stop here and adapt the preparation script to the inspected layout.


In [ ]:
# Example only — execute after confirming the folder layout.
# !python training/prepare_ffpp.py \
#   --dataset-root /content/unmaskd_data/video_raw \
#   --output-dir /content/unmaskd_data/video_frames \
#   --frames-per-video 8


## 5. Train the first video baseline

Once `/content/unmaskd_data/video_frames` is an ImageFolder dataset with `train/val/test` and `real/fake` classes, run the baseline. The script reports real metrics from the held-out test set and saves a checkpoint.


In [ ]:
from pathlib import Path
data_dir = Path('/content/unmaskd_data/video_frames')
if data_dir.exists():
    print('Prepared dataset found:', data_dir)
    print('Subdirectories:', [p.name for p in data_dir.iterdir() if p.is_dir()])
else:
    print('Prepared dataset not found yet. Complete the verified preparation step first.')


In [ ]:
# Start with a small run to validate the pipeline. Increase epochs/batch size after it works.
# !python training/train_video.py \
#   --data-dir /content/unmaskd_data/video_frames \
#   --epochs 5 \
#   --batch-size 32


## 6. Audio dataset — ASVspoof 2021 DF

ASVspoof 2021's Speech Deepfake (DF) task contains bona-fide and spoofed speech. Registration and dataset access are handled by ASVspoof/Zenodo. Download it separately and keep it outside GitHub.

After download, create a manifest with: `path`, `label` (`bonafide`/`spoof`), `speaker/source_id`, and `split`. Use speaker/source-disjoint splits.


In [ ]:
# Placeholder for the ASVspoof DF path after you download and extract it.
AUDIO_ROOT = Path('/content/unmaskd_data/asvspoof_df')
print('Expected audio dataset location:', AUDIO_ROOT)
print('Next stage: build the audio manifest and train the anti-spoof model.')


## 7. Save training artifacts

Keep checkpoints and metrics in `/content/unmaskd_artifacts` during training. Upload only selected artifacts/configs to GitHub or approved storage. Never upload raw datasets or `kaggle.json`.


In [ ]:
!mkdir -p /content/unmaskd_artifacts
print('Artifact directory ready:', '/content/unmaskd_artifacts')


## Target Unmaskd live-call pipeline

Video detector + audio anti-spoof detector → calibrated probabilities → signal-quality checks → temporal smoothing → validated multimodal fusion → authenticity score.

The live UI currently uses short rolling media windows. A production model should be trained with compression, frame drops, blur, resizing, lighting variation, microphone noise, echo/reverb, resampling and other WebRTC/VoIP-like conditions.
